In [ ]:
# Importing libraries
import os
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
# Above is a special style template for matplotlib, highly useful for visualizing time series data
%matplotlib inline
from pylab import rcParams
from plotly import tools
import chart_studio.plotly as py
from plotly.offline import init_notebook_mode, iplot
init_notebook_mode(connected=True)
import plotly.graph_objs as go
import plotly.figure_factory as ff
import statsmodels.api as sm
from numpy.random import normal, seed
from scipy.stats import norm
from statsmodels.tsa.arima_model import ARMA
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima_process import ArmaProcess
from statsmodels.tsa.arima_model import ARIMA
import math
from sklearn.metrics import mean_squared_error
print(os.listdir("../input"))

In [ ]:
import pandas as pd
GOOGL = pd.read_csv('../input/GOOGL_2006-01-01_to_2018-01-01.csv', sep=',', na_values=[''], quotechar='"', index_col="Date", parse_dates=['Date'])
GOOGL.head()

In [ ]:
GOOGL["diff_1"] = GOOGL["Close"].diff(1)
GOOGL["diff_2"] = GOOGL["diff_1"].diff(1)

In [ ]:
plt.plot(GOOGL["diff_1"])
plt.show()

In [ ]:
plt.plot(GOOGL["diff_2"])
plt.show()

In [ ]:
plot_pacf(GOOGL["diff_1"])
plt.show()

In [ ]:
sub = GOOGL.loc["2006-01":"2018-01","Close"]
train = sub.loc["2006-01":"2014-01"]
test = sub.loc["2014-01":"2018-01"]

In [ ]:
plt.figure(figsize=(12,8))
plt.plot(train)
plt.show()

In [ ]:
train_diff_1 = train.diff(1).dropna()
from statsmodels.tsa.stattools import adfuller as ADF
train_adf = ADF(train)
train_diff_1_adf = ADF(train_diff_1)
print(train_adf)
print(train_diff_1_adf)


In [ ]:
train_results = sm.tsa.arma_order_select_ic(train_diff_1, ic=['aic', 'bic'], trend='n', max_ar=5, max_ma=5)

print('AIC', train_results.aic_min_order)
print('BIC', train_results.bic_min_order)

In [ ]:
p=0
q=0
d=1
from statsmodels.tsa.arima.model import ARIMA
model = ARIMA(train, order=(p, d, q))
result = model.fit()
resid = result.resid
plot_acf(resid,lags=20)
plt.show()

In [ ]:
predict = result.predict(dynamic=False)
fig,ax= plt.subplots(1,2,figsize=(10, 8))
ax[0].plot(predict)
ax[1].plot(train)
plt.tight_layout
plt.show()


In [ ]:
predict = result.predict(dynamic=False)
plt.subplots(figsize=(10, 8))
plt.plot(predict,color='blue',linestyle='dashed')
plt.plot(train,color='red',linestyle='--')

plt.show()


In [ ]:
predict = result.predict(dynamic=False)

fig, ax = plt.subplots(figsize=(12, 6))

train.plot(
    ax=ax,
    label="Actual",
    linewidth=2,
    alpha=0.8
)

predict.plot(
    ax=ax,
    label="In-sample Prediction",
    linewidth=1.5,
    alpha=0.8
)

ax.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
ax = sub.plot(ax=ax)
#预测数据
pred.plot(ax=ax)
plt.show()

In [ ]:
resid

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
import pandas as pd

history = train.copy()
preds = []

for i in range(len(test)):
    model = ARIMA(history, order=(0, 1, 0))
    result = model.fit()

    yhat = result.forecast(steps=1).iloc[0]
    preds.append(yhat)

    # 把 test 中已经发生的真实值加入 history
    history = pd.concat([history, test.iloc[i:i+1]])

pred_rolling = pd.Series(preds, index=test.index)
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

#plt.plot(train.index, train, label="Train",linewidth=1.5,alpha=0.8)
plt.plot(test.index, test, label="Test",linewidth=3,alpha=0.8)
plt.plot(pred_rolling.index, pred_rolling, label="Rolling Forecast",linewidth=1.5,alpha=0.8)

plt.legend()
plt.show()

In [ ]:
result.resid

In [67]:
from statsmodels.tsa.arima.model import ARIMA

model = ARIMA(train, order=(1, 1, 1))
result = model.fit()

print(result.summary())

                               SARIMAX Results                                
Dep. Variable:                  Close   No. Observations:                 2033
Model:                 ARIMA(1, 1, 1)   Log Likelihood               -6244.237
Date:                Wed, 13 May 2026   AIC                          12494.474
Time:                        11:46:27   BIC                          12511.324
Sample:                             0   HQIC                         12500.656
                               - 2033                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1         -0.7077      0.755     -0.937      0.349      -2.188       0.773
ma.L1          0.6982      0.764      0.914      0.361      -0.800       2.196
sigma2        27.3350      0.301     90.732      0.0

In [68]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(test, pred_rolling)
rmse = np.sqrt(mean_squared_error(test, pred_rolling))

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 6.678430983118173
RMSE: 9.544420738284446


In [74]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

model = ExponentialSmoothing(
    train,
    trend="add",
    seasonal=None
)

result = model.fit()
test = test.dropna()
forecast = result.forecast(steps=len(test))
forecast = pd.Series(forecast.values, index=test.index)

mae = mean_absolute_error(test, forecast)
rmse = np.sqrt(mean_squared_error(test, forecast))

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 91.57580373088534
RMSE: 115.46200262591866


In [77]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

model = ExponentialSmoothing(
    train,
    trend="add",
    seasonal="add",
    seasonal_periods=90,
)

result = model.fit()
test = test.dropna()
forecast = result.forecast(steps=len(test))
forecast = pd.Series(forecast.values, index=test.index)

mae = mean_absolute_error(test, forecast)
rmse = np.sqrt(mean_squared_error(test, forecast))

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 73.35767069080751
RMSE: 92.40863642002968
